# 2 · Do you want to build a snowman? Prepare the data yourself

**What you practise (from the feature-engineering lecture):**
- Planning the preparation **per column** before writing code.
- Imputing, scaling and one-hot encoding with fitted objects.
- Building a clean `X_train_fe` (**M1**).
- Handling **new rows**, including a category the encoder has never seen (**M2**), and the product decision behind it.
- The same steps as **one recipe**: `Pipeline` + `ColumnTransformer`.

## ⛄ The case: SnowDay

**SnowDay** is a family app for snow days: sledging spots, hot-chocolate recipes, and a "snowman log" where kids record every snowman they build.

**Jamie**, the PM, has a fun feature idea: *"Before the kids go out, the app predicts how tall today's snowman will get."*

The data scientist has 14 logged snowmen. Before any model can learn, the table must be **model-ready**. That's your job.

<img src="../assets/olaf.jpeg" width="220" alt="Snowman illustration">

> 📌 14 rows is far too few to *judge* a model. This notebook is **preparation only**: no model score from these examples. (And no, lunch probably doesn't make snowmen taller. It's a practice table.)

> **How this notebook works**
> - Run the cells **from top to bottom** with `Shift + Enter`.
> - ▶️ **Run** cells work as they are.
> - ✏️ **Your turn** cells contain `___` (three underscores). Replace each `___` with your answer, then run the cell.
> - ✅ **Check** cells tell you whether you got it right. If a check fails, read the hint, fix the cell above and run both again.
> - 🧠 **Think** cells ask a PM question. Double-click the answer cell below it and write 1–3 sentences.
> - If you see `NameError: name '___' is not defined`, a blank is still empty.
> - Stuck? The `solutions/` folder has the same notebook, solved.

## Step 0 · Load the tools and the snowmen

In [1]:
# ▶️ Run: libraries and the 14 logged snowmen
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

snowmen = pd.DataFrame({
    "temp": [-3, 5, 0, 7, 3, -1, 1, None, -6, 3, 0, -1, None, -2],
    "lunch": ["soup", "sandwich", "soup", "burger", "sandwich", "soup", "cereal",
              "salad", "sandwich", "burger", "soup", "cereal", "burger", "soup"],
    "dinner": ["pizza", "pizza", "noodles", None, "fishsticks", "pizza", None,
               "fishsticks", "noodles", "pizza", None, "pizza", "fishsticks", "pizza"],
    "precipitation": ["yes", "no", "yes", "yes", "yes", "yes", "no",
                      "yes", "yes", "no", "yes", "yes", "yes", "no"],
    "height_snowman_cm": [100, 0, 75, 0, 20, 25, 0, 35, 170, 0, 85, 85, 45, 0],
}).fillna(np.nan)
snowmen

,temp,lunch,dinner,precipitation,height_snowman_cm
0,-3.0,soup,pizza,yes,100
1,5.0,sandwich,pizza,no,0
2,0.0,soup,noodles,yes,75
3,7.0,burger,NaN,yes,0
4,3.0,sandwich,fishsticks,yes,20
5,-1.0,soup,pizza,yes,25
6,1.0,cereal,NaN,no,0
7,NaN,salad,fishsticks,yes,35
8,-6.0,sandwich,noodles,yes,170
9,3.0,burger,pizza,no,0


| column | meaning |
|---|---|
| `temp` | temperature in °C that day |
| `lunch`, `dinner` | what the builder ate |
| `precipitation` | did it snow/rain that day? (yes/no) |
| **`height_snowman_cm`** | **target**: how tall the snowman got |

## Step 1 · Separate inputs and target

In [2]:
# ✏️ Your turn: which column is the target?
target = "height_snowman_cm"
X_train = snowmen.drop(columns=target)
y_train = snowmen[target]
X_train.shape

(14, 4)

In [3]:
# ✅ Check
assert X_train.shape == (14, 4) and target not in X_train.columns, "The target is the snowman height."
print("✅ Correct: 14 snowmen, 4 input columns. All 14 rows are our training data.")

✅ Correct: 14 snowmen, 4 input columns. All 14 rows are our training data.


## Step 2 · Plan before you code

For every column: what **type** is it, how many **gaps**, and which **tools** (in which order)?
Types: `"numeric"`, `"nominal"` (words without order) or `"binary"` (exactly two values).

In [4]:
# ▶️ Run: count the gaps
X_train.isna().sum()

temp             2
lunch            0
dinner           3
precipitation    0
dtype: int64

In [5]:
# ✏️ Your turn: complete the plan
plan = {
    "temp":          {"type": "numeric", "gaps": 2, "tools": "median imputer → scaler"},
    "lunch":         {"type": "nominal", "gaps": 0, "tools": "one-hot"},
    "dinner":        {"type": "nominal", "gaps": 3, "tools": "most-frequent imputer → one-hot"},
    "precipitation": {"type": "binary",  "gaps": 0, "tools": "one-hot, drop first"},
}
pd.DataFrame(plan).T

,type,gaps,tools
temp,numeric,2,median imputer → scaler
lunch,nominal,0,one-hot
dinner,nominal,3,most-frequent imputer → one-hot
precipitation,binary,0,"one-hot, drop first"


In [6]:
# ✅ Check
assert [plan[c]["type"] for c in plan] == ["numeric", "nominal", "nominal", "binary"], "Words without order are nominal; yes/no is binary."
assert plan["temp"]["gaps"] == 2 and plan["dinner"]["gaps"] == 3, "Look at the gap counts above."
print("✅ Plan complete. If the plan is right, the code is just typing.")

✅ Plan complete. If the plan is right, the code is just typing.


## Step 3 · `temp`: fill the gaps, then scale

Order matters: a scaler can't compute a mean with empty cells, so **impute first, then scale**.

In [7]:
# ✏️ Your turn: fill temp with its median (learned from these training rows)
temp_imputer = SimpleImputer(strategy="median").set_output(transform="pandas")
temp_filled = temp_imputer.fit_transform(X_train[["temp"]])
print("Learned median:", temp_imputer.statistics_[0])

Learned median: 0.0


In [8]:
# ✅ Check
assert temp_imputer.statistics_[0] == 0.0 and temp_filled["temp"].isna().sum() == 0, "Use strategy='median'."
print("✅ Correct: the median of the 12 known temperatures is 0.0 °C.")

✅ Correct: the median of the 12 known temperatures is 0.0 °C.


🧠 **Think:** the median happens to be 0 °C. Isn't that the same as "filling with zero", which we said is bad?

**✍️ Sample answer:** No, it's a coincidence. The 0.0 was **learned** from the training temperatures (the middle of −6 … 7). With warmer training data the median would be different, while a hard-coded 0 would not change. The rule is about meaning: never write 0 just because a cell is empty.

In [9]:
# ✏️ Your turn: scale the FILLED temperatures (fit on training rows, apply to the same rows)
temp_scaler = StandardScaler().set_output(transform="pandas")
temp_ready = temp_scaler.fit_transform(temp_filled)
print("mean:", abs(round(temp_ready["temp"].mean(), 2)), "| SD:", round(temp_ready["temp"].std(ddof=0), 2))

mean: 0.0 | SD: 1.0


## Step 4 · `dinner`: fill the gaps with the most frequent dinner

In [10]:
# ✏️ Your turn: which strategy works for words?
dinner_imputer = SimpleImputer(strategy="most_frequent").set_output(transform="pandas")
dinner_filled = dinner_imputer.fit_transform(X_train[["dinner"]])
print("Learned fill value:", dinner_imputer.statistics_[0])

Learned fill value: pizza


In [11]:
# ✅ Check
assert dinner_imputer.statistics_[0] == "pizza", "You can't average pizza and noodles: use 'most_frequent'."
print("✅ Correct: pizza (6 times) is the most frequent dinner.")

✅ Correct: pizza (6 times) is the most frequent dinner.


## Step 5 · One-hot encode the words

With `drop="first"`, the first category (alphabetically) is dropped and becomes the "all zeros" **reference**.

In [12]:
# ✏️ Your turn: encode lunch and precipitation together, and the FILLED dinner separately
lunch_precip_encoder = OneHotEncoder(drop="first", sparse_output=False).set_output(transform="pandas")
lunch_precip_ready = lunch_precip_encoder.fit_transform(X_train[["lunch", "precipitation"]])

dinner_encoder = OneHotEncoder(drop="first", sparse_output=False).set_output(transform="pandas")
dinner_ready = dinner_encoder.fit_transform(dinner_filled)

print(list(lunch_precip_ready.columns))
print(list(dinner_ready.columns))

['lunch_cereal', 'lunch_salad', 'lunch_sandwich', 'lunch_soup', 'precipitation_yes']
['dinner_noodles', 'dinner_pizza']


In [13]:
# ✅ Check
assert list(lunch_precip_ready.columns) == ["lunch_cereal", "lunch_salad", "lunch_sandwich", "lunch_soup", "precipitation_yes"]
assert list(dinner_ready.columns) == ["dinner_noodles", "dinner_pizza"], "Encode dinner_filled (the imputed column), not the raw one."
print("✅ Correct: burger and fishsticks are the references (all zeros). precipitation became one 0/1 column.")

✅ Correct: burger and fishsticks are the references (all zeros). precipitation became one 0/1 column.


## Step 6 · Glue it together: **milestone M1** 🏁

In [14]:
# ✏️ Your turn: put the three ready pieces side by side (axis=1 means: as columns)
X_train_fe = pd.concat([temp_ready, dinner_ready, lunch_precip_ready], axis=1)
X_train_fe

,temp,dinner_noodles,dinner_pizza,lunch_cereal,lunch_salad,lunch_sandwich,lunch_soup,precipitation_yes
0,-1.078720,0.0,1.0,0.0,0.0,0.0,1.0,1.0
1,1.438293,0.0,1.0,0.0,0.0,1.0,0.0,0.0
2,-0.134840,1.0,0.0,0.0,0.0,0.0,1.0,1.0
3,2.067546,0.0,1.0,0.0,0.0,0.0,0.0,1.0
4,0.809040,0.0,0.0,0.0,0.0,1.0,0.0,1.0
5,-0.449467,0.0,1.0,0.0,0.0,0.0,1.0,1.0
6,0.179787,0.0,1.0,1.0,0.0,0.0,0.0,0.0
7,-0.134840,0.0,0.0,0.0,1.0,0.0,0.0,1.0
8,-2.022600,1.0,0.0,0.0,0.0,1.0,0.0,1.0
9,0.809040,0.0,1.0,0.0,0.0,0.0,0.0,0.0


In [15]:
# ✅ Check (M1)
assert X_train_fe.shape == (14, 8), "Expected 14 rows × 8 columns."
assert X_train_fe.isna().sum().sum() == 0, "There are still gaps."
assert X_train_fe.index.equals(y_train.index), "Rows must still line up with y_train."
assert target not in X_train_fe.columns
print("🏁 M1 reached: 14 snowmen × 8 columns, only numbers, no gaps, rows match y_train.")

🏁 M1 reached: 14 snowmen × 8 columns, only numbers, no gaps, rows match y_train.


## Step 7 · New snowmen arrive: **milestone M2** 🏁

Two new logs come in. We prepare them with the objects we **already fitted**: `transform` only, never `fit`.

In [16]:
# ▶️ Run: two new snowmen. The second one's builder ate a WRAP, a lunch the encoder has never seen.
X_new = pd.DataFrame({
    "temp": [np.nan, 2.0],
    "lunch": ["soup", "wrap"],
    "dinner": [np.nan, "pizza"],
    "precipitation": ["yes", "no"],
}, index=["known", "new_category"])
X_new

,temp,lunch,dinner,precipitation
known,NaN,soup,NaN,yes
new_category,2.0,wrap,pizza,no


In [17]:
# ✏️ Your turn: prepare the KNOWN row with the fitted objects. Which method?
known = X_new.loc[["known"]]
known_temp = temp_scaler.transform(temp_imputer.transform(known[["temp"]]))
known_dinner = dinner_encoder.transform(dinner_imputer.transform(known[["dinner"]]))
known_other = lunch_precip_encoder.transform(known[["lunch", "precipitation"]])
known_fe = pd.concat([known_temp, known_dinner, known_other], axis=1)
known_fe

,temp,dinner_noodles,dinner_pizza,lunch_cereal,lunch_salad,lunch_sandwich,lunch_soup,precipitation_yes
known,-0.13484,0.0,1.0,0.0,0.0,0.0,1.0,1.0


In [18]:
# ✅ Check
assert known_fe.columns.equals(X_train_fe.columns), "Same columns, same order as X_train_fe."
assert abs(known_fe.loc["known", "temp"] - temp_ready["temp"][temp_filled["temp"] == 0].iloc[0]) < 1e-9
assert known_fe.loc["known", "dinner_pizza"] == 1 and known_fe.loc["known", "lunch_soup"] == 1
print("✅ Correct: missing temp → learned median 0.0 → scaled with the training ruler; missing dinner → pizza.")

✅ Correct: missing temp → learned median 0.0 → scaled with the training ruler; missing dinner → pizza.


Now the **wrap**. Before you run the next cell, **predict**: what will the fitted encoder do with `lunch = "wrap"`?

A · stop with an error  B · learn "wrap" as a new column  C · guess the closest lunch  D · use the most frequent lunch

In [19]:
# ✏️ Your turn: your prediction ("A", "B", "C" or "D"), BEFORE running the next cell
my_prediction = "A"

In [20]:
# ▶️ Run: observe
try:
    lunch_precip_encoder.transform(X_new.loc[["new_category"], ["lunch", "precipitation"]])
except ValueError as error:
    print("Expected error:", error)
print("\nYou predicted:", my_prediction, "→ the answer is A: a fitted encoder never learns during transform.")

Expected error: Found unknown categories ['wrap'] in column 0 during transform

You predicted: A → the answer is A: a fitted encoder never learns during transform.


That error is **not you breaking the notebook**. The encoder knows exactly 5 lunches. Learning a new one would mean a new column the model has no weight for. New categories need **retraining** (encoder and model together).

There is a setting that keeps the system running instead: `handle_unknown="ignore"` gives the unknown lunch **0 in every lunch column**. But watch what that means together with `drop="first"`:

In [21]:
# ▶️ Run: two "ignore" encoders: one WITH drop="first", one keeping ALL columns
both_lunches = pd.DataFrame({"lunch": ["burger", "wrap"]}, index=["burger", "wrap"])
with_drop = OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False).set_output(transform="pandas")
keep_all = OneHotEncoder(handle_unknown="ignore", sparse_output=False).set_output(transform="pandas")
with_drop.fit(X_train[["lunch"]]); keep_all.fit(X_train[["lunch"]])

import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore")      # scikit-learn warns about the unknown category; we look at the result
    print("WITH drop='first':"); print(with_drop.transform(both_lunches), "\n")
print("KEEP all columns:"); print(keep_all.transform(both_lunches))

WITH drop='first':
        lunch_cereal  lunch_salad  lunch_sandwich  lunch_soup
burger           0.0          0.0             0.0         0.0
wrap             0.0          0.0             0.0         0.0 

KEEP all columns:
        lunch_burger  lunch_cereal  lunch_salad  lunch_sandwich  lunch_soup
burger           1.0           0.0          0.0             0.0         0.0
wrap             0.0           0.0          0.0             0.0         0.0


In [22]:
# ✏️ Your turn: with drop="first" + ignore, which KNOWN lunch does the wrap look exactly like?
wrap_looks_like = "burger"

In [23]:
# ✅ Check (M2)
assert wrap_looks_like.strip().lower() == "burger", "Compare the two rows of the WITH drop='first' table."
print("🏁 M2 reached. With drop='first', 'all zeros' already means burger, so the wrap is silently scored as a burger.")
print("   Keeping all columns, a wrap is all zeros = 'none of the lunches I know'. Still visible.")

🏁 M2 reached. With drop='first', 'all zeros' already means burger, so the wrap is silently scored as a burger.
   Keeping all columns, a wrap is all zeros = 'none of the lunches I know'. Still visible.


🧠 **Think (PM decision):** SnowDay launches in Italy next month, and kids will log lunches like `pasta`. Should the prediction **fail loudly** (default error) or **keep scoring with "unknown lunch" and alert someone**? Who should get the alert?

**✍️ Sample answer:** Keep scoring with `handle_unknown="ignore"` and **all columns kept** (so unknown ≠ burger), but make it visible: count unknown lunches per day, alert the model owner (and the Italy launch team) above a threshold, and retrain once enough Italian logs exist. A common middle ground: allow and alert, but fail if more than X% of rows are unknown. Ideally the launch team tells the data team *before* the launch.

## Step 8 · The same steps as one recipe: `Pipeline` + `ColumnTransformer`

You just did 3 imputers/scalers/encoders by hand, and had to remember to call `transform` (not `fit`) on the new rows, in the right order. In production that's a chance to slip every day. A recipe object does it for you:

- **`Pipeline`** = steps **one after another** on the same columns (fill → scale).
- **`ColumnTransformer`** = **different columns, different stations**, side by side, each column **exactly once**.

### 8a · Spot the bug first 🐞
A colleague put "fill" and "scale" as two **separate stations** for the same column:

In [24]:
# ▶️ Run: the classic mistake: two branches for the same column
buggy = ColumnTransformer([
    ("fill", SimpleImputer(strategy="median"), ["temp"]),
    ("scale", StandardScaler(), ["temp"]),
]).set_output(transform="pandas")
buggy.fit_transform(X_train).head(8)

,fill__temp,scale__temp
0,-3.0,-1.021055
1,5.0,1.312785
2,0.0,-0.145865
3,7.0,1.896245
4,3.0,0.729325
5,-1.0,-0.437595
6,1.0,0.145865
7,0.0,NaN


Two `temp` columns: one filled but **not scaled**, one scaled but **still with gaps** (NaN). Branches don't feed into each other; each takes the original column. **Fix:** put fill → scale into **one** `Pipeline`, and give that pipeline the column **once**.

### 8b · The correct recipe

In [25]:
# ✏️ Your turn: complete the recipe. Each input column must appear in exactly ONE branch.
number_steps = Pipeline([
    ("fill", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])
word_steps = Pipeline([
    ("fill", SimpleImputer(strategy="most_frequent")),
    ("one_hot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),   # all columns kept: unknown ≠ burger
])
recipe = ColumnTransformer([
    ("numbers", number_steps, ["temp"]),
    ("words", word_steps, ["lunch", "dinner", "precipitation"]),
]).set_output(transform="pandas")

X_train_recipe = recipe.fit_transform(X_train)     # ONE fit learns the median, the scale, pizza AND all categories
print(X_train_recipe.shape)
X_train_recipe.head()

(14, 11)


,numbers__temp,words__lunch_burger,words__lunch_cereal,words__lunch_salad,words__lunch_sandwich,words__lunch_soup,words__dinner_fishsticks,words__dinner_noodles,words__dinner_pizza,words__precipitation_no,words__precipitation_yes
0,-1.078720,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0
1,1.438293,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0
2,-0.134840,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0
3,2.067546,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
4,0.809040,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0


In [26]:
# ✅ Check
assert X_train_recipe.shape == (14, 11), "Expected 14 × 11 (1 temp + 3 dinners + 5 lunches + 2 precipitation)."
assert X_train_recipe.isna().sum().sum() == 0
print("✅ One object, one fit. 11 columns because no category is dropped this time.")

✅ One object, one fit. 11 columns because no category is dropped this time.


In [27]:
# ▶️ Run: new rows, including the wrap. Only transform. No error, no refitting, no hand-typed medians.
import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    print(recipe.transform(X_new).filter(like="lunch"))

              words__lunch_burger  words__lunch_cereal  words__lunch_salad  \
known                         0.0                  0.0                 0.0   
new_category                  0.0                  0.0                 0.0   

              words__lunch_sandwich  words__lunch_soup  
known                           0.0                1.0  
new_category                    0.0                0.0  


🧠 **Think:** as a PM you'll never write this code. What **one question** would you ask your data scientist about it, and why?

**✍️ Sample answer:** *"Is all of the preprocessing inside the pipeline that gets saved with the model?"* If yes, production fills, scales and encodes new data with exactly the numbers learned in training (no training–serving skew, no hand-copied medians, no accidental refitting). What it can't guarantee: that the raw column still *means* the same thing upstream. That's what the feature spec and monitoring are for (notebook 3).

## ✅ Check your understanding

1. Why should new rows use `transform` instead of `fit_transform`?
2. Why could encoding meals as 0, 1, 2 mislead a linear or distance-based model?
3. With `drop="first"` and `handle_unknown="ignore"`, what happens to an unknown lunch, and why is that risky?

**Your answers to 1–3:**

**✍️ Sample answer:** (1) `transform` reuses the training statistics; refitting would let the new rows change the preparation, and the model would receive inputs on a different ruler than it learned on. (2) The codes invent an order and distances between meals, which the model treats as real. (3) It becomes all zeros, which is exactly how the reference category (burger) looks, so it is silently scored as a burger. Keeping all columns (or failing loudly) keeps it visible.

### 🎉 Well done! Next: notebook 3, *a 'perfect' churn model lands on your desk.*